## Data Reading

In [0]:
df = spark.read.format('parquet')\
        .option('inferSchema',True)\
        .load('abfss://bronze@slcardatalake.dfs.core.windows.net/rawdata/')

In [0]:
display(df)

## Data Transformation

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn('Model_category',split(col('Model_ID'),'-')[0])
display(df)

In [0]:
df.withColumn('Units_sold',col('Units_Sold').cast(StringType()))

In [0]:
df = df.withColumn('RevperUnit',col('Revenue')/col("Units_Sold"))
display(df)

In [0]:
display(df.groupBy('Year','BranchName').agg(sum("Units_Sold").alias('Total_units')).sort('Year','Total_units',ascending=[1,0]))

Databricks visualization. Run in Databricks to view.

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@slcardatalake.dfs.core.windows.net/carsales')\
    .save()

### Querying silver table

In [0]:
%sql
Select * from parquet.`abfss://silver@slcardatalake.dfs.core.windows.net/carsales`